# Results Tables

Reproduces the three modality results tables in the paper's *Results* section
(`tab:vision_results`, `tab:tabular_results`, `tab:text_results`) directly from
`experiment_records.jsonl`.

For each dataset, macro-F1 is summarized as **mean ± std over all 60 runs**
(20 test sizes × 3 trials) for four prediction-only methods:

- **Experts (W)** — the proposed expert-weighted aggregation (`experts_weighted`)
- **All (Maj)** — majority voting over the full pool (`all_majority`)
- **EM** — Dawid–Skene EM (`em`)
- **SML** — Spectral Meta-Learner (`sml`)

The best mean per dataset is marked. No predictions or raw data are required —
every record already carries a `macro_f1` dict keyed by method.

In [ ]:
import json
import numpy as np
import pandas as pd
from collections import defaultdict

RESULTS_JSONL = "data/results/experiment_records.jsonl"

# method key in the records -> column label in the paper tables
METHODS = {
    "experts_weighted": "Experts (W)",
    "all_majority":     "All (Maj)",
    "em":               "EM",
    "sml":              "SML",
}

# raw record dataset name -> (modality, pretty label used in the paper)
DATASETS = {
    # vision
    "CIFAR10":      ("vision", "CIFAR-10"),
    "CIFAR100":     ("vision", "CIFAR-100"),
    "SVHN":         ("vision", "SVHN"),
    "ImageWoof-49": ("vision", "ImageWoof"),
    "Food-101":     ("vision", "Food-101"),
    "Oxford-IIIT Pet (Binary: Cats vs Dogs)": ("vision", "Oxford Pet"),
    "PCAM":         ("vision", "PCam"),
    "Kuzushiji-49": ("vision", "Kuzushiji-49"),
    # tabular
    "sklearn_covertype": ("tabular", "Covertype"),
    "HEPMASS":           ("tabular", "HepMass"),
    "HIGGS":             ("tabular", "HIGGS"),
    "Santander_Customer_Transaction_Prediction": ("tabular", "Santander CT"),
    "Satellite Image":   ("tabular", "Satellite Image"),
    "uci_letter":        ("tabular", "UCI Letter"),
    "UCI_Pendigits":     ("tabular", "UCI Pendigits"),
    "Banknote_Authentication": ("tabular", "Banknote Auth."),
    # text
    "20Newsgroups":    ("text", "20 NG"),
    "Amazon_Polarity": ("text", "Amazon Pol."),
    "CoLA":            ("text", "CoLA"),
    "IMDb":            ("text", "IMDb"),
    "MultiNLI":        ("text", "MultiNLI"),
    "QNLI":            ("text", "QNLI"),
    "SST2":            ("text", "SST-2"),
    "DBpedia":         ("text", "DBpedia"),
}

# order datasets within each modality exactly as the paper lists them
ORDER = {
    "vision":  ["CIFAR-10", "CIFAR-100", "SVHN", "ImageWoof", "Food-101",
                "Oxford Pet", "PCam", "Kuzushiji-49"],
    "tabular": ["Covertype", "HepMass", "HIGGS", "Santander CT", "Satellite Image",
                "UCI Letter", "UCI Pendigits", "Banknote Auth."],
    "text":    ["20 NG", "Amazon Pol.", "CoLA", "IMDb", "MultiNLI",
                "QNLI", "SST-2", "DBpedia"],
}

## Aggregate macro-F1 per dataset

Collect each method's macro-F1 across the 60 runs of every dataset, then take
the mean and standard deviation. `macro_f1` in each record is a dict keyed by
method name.

In [ ]:
# dataset -> method -> list of per-run macro-F1
vals = defaultdict(lambda: defaultdict(list))
n_records = defaultdict(int)
with open(RESULTS_JSONL) as f:
    for line in f:
        rec = json.loads(line)
        ds = rec["dataset"]
        mf = rec.get("macro_f1", {})
        if not isinstance(mf, dict):
            continue
        n_records[ds] += 1
        for key in METHODS:
            if key in mf and mf[key] is not None:
                vals[ds][key].append(float(mf[key]))

print(f"datasets found: {len(vals)}")
bad = {ds: n for ds, n in n_records.items() if n != 60}
if bad:
    print("WARNING: datasets without exactly 60 records:", bad)
missing_map = sorted(set(vals) - set(DATASETS))
if missing_map:
    print("WARNING: records with no DATASETS mapping:", missing_map)

## Build and print the three tables

One table per modality, columns `Experts (W) | All (Maj) | EM | SML`, each cell
`mean ± std`. The best mean in each row is marked with `*`.

In [ ]:
def cell(mean, std, is_best):
    s = f"{mean:.4f} ± {std:.4f}"
    return f"*{s}*" if is_best else f" {s} "

def build_modality_table(modality):
    rows = []
    for pretty in ORDER[modality]:
        raw = next(r for r, (m, p) in DATASETS.items() if p == pretty)
        means = {k: np.mean(vals[raw][k]) for k in METHODS if vals[raw][k]}
        stds  = {k: np.std(vals[raw][k]) for k in METHODS if vals[raw][k]}
        best_key = max(means, key=means.get)
        row = {"Dataset": pretty}
        for key, label in METHODS.items():
            row[label] = cell(means[key], stds[key], key == best_key)
        rows.append(row)
    return pd.DataFrame(rows)

for modality in ["vision", "tabular", "text"]:
    print("=" * 78)
    print(f"{modality.upper()}  (macro-F1, mean ± std over trials and sizes; * = best)")
    print("=" * 78)
    tbl = build_modality_table(modality)
    print(tbl.to_string(index=False))
    print()

## Win count

How many of the 24 datasets each method wins (best mean macro-F1), and the
Experts (W) win count by modality — the headline numbers in the outcome summary.

In [ ]:
win_total = defaultdict(int)
win_by_modality = defaultdict(lambda: defaultdict(int))
for raw, (modality, pretty) in DATASETS.items():
    means = {k: np.mean(vals[raw][k]) for k in METHODS if vals[raw][k]}
    best_key = max(means, key=means.get)
    win_total[METHODS[best_key]] += 1
    win_by_modality[modality][METHODS[best_key]] += 1

print("Wins across all 24 datasets:")
for label in METHODS.values():
    print(f"  {label:12s}: {win_total[label]}")

ew = "Experts (W)"
print(f"\nExperts (W) wins by modality:")
for modality in ["vision", "tabular", "text"]:
    print(f"  {modality:8s}: {win_by_modality[modality][ew]} / 8")
print(f"\nExperts (W) total: {win_total[ew]} / 24")